In [1]:
import requests
from bs4 import BeautifulSoup
import feedparser
import os
import re
import time


In [6]:
feed_url = "https://shortfictionbreak.com/category/genre/thrillersuspense/feed/"
feed = feedparser.parse(feed_url)
urls = [entry.link for entry in feed.entries]
print(f"Found {len(urls)} story URLs")
urls  # preview a few

Found 10 story URLs


['https://shortfictionbreak.com/the-weight-of-silence/',
 'https://shortfictionbreak.com/the-bargain-2/',
 'https://shortfictionbreak.com/waiting-for-a-call/',
 'https://shortfictionbreak.com/the-voice-2/',
 'https://shortfictionbreak.com/the-mallard/',
 'https://shortfictionbreak.com/not-the-gift-of-the-magi/',
 'https://shortfictionbreak.com/unbound-3/',
 'https://shortfictionbreak.com/the-ghost/',
 'https://shortfictionbreak.com/a-study-in-cashmere/',
 'https://shortfictionbreak.com/duck-eggs/']

In [12]:
thriller_feed = "https://shortfictionbreak.com/category/genre/thrillersuspense/feed/"
horror_feed   = "https://shortfictionbreak.com/category/genre/horror/feed/"
mystery_feed = "https://shortfictionbreak.com/category/genre/mystery/feed/"
science_fiction_feed = "https://shortfictionbreak.com/category/genre/science-fiction/feed/"
feeds = [thriller_feed, horror_feed, mystery_feed, science_fiction_feed]
urls = []

categories = [
    "thrillersuspense",
    "horror",
    "mystery",
    "science-fiction"
]

BASE = "https://shortfictionbreak.com/category/genre/{}/page/{}/"
urls = []

for cat in categories:
    print(f"\nCrawling category: {cat}")
    page = 1
    while True:
        url = BASE.format(cat, page)
        r = requests.get(url)
        if r.status_code != 200:
            break
        soup = BeautifulSoup(r.text, "html.parser")
        links = [a["href"] for a in soup.select("h2.entry-title a")]
        if not links:
            break
        print(f"Page {page}: {len(links)} stories")
        urls.extend(links)
        page += 1
        time.sleep(1.5)
print(f"\nTotal unique stories: {len(set(urls))}")




Crawling category: thrillersuspense
Page 1: 30 stories
Page 2: 30 stories
Page 3: 16 stories

Crawling category: horror
Page 1: 30 stories
Page 2: 30 stories
Page 3: 1 stories

Crawling category: mystery
Page 1: 13 stories

Crawling category: science-fiction
Page 1: 30 stories
Page 2: 26 stories

Total unique stories: 183


In [13]:
def clean_text(text):
    """Basic whitespace + dash cleanup."""
    text = re.sub(r"\s+", " ", text)
    text = text.replace("–", "-").strip()
    return text

def scrape_story(url, out_dir="stories"):
    """Download one story from Short Fiction Break."""
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        title_tag = soup.find("h1", class_="entry-title")
        content_tag = soup.find("div", class_="entry-content")

        if not content_tag:
            print(f"No content found: {url}")
            return None

        title = title_tag.get_text(strip=True) if title_tag else "untitled"
        story = clean_text(content_tag.get_text(separator="\n", strip=True))

        os.makedirs(out_dir, exist_ok=True)
        filename = re.sub(r"[^a-zA-Z0-9_-]", "_", title)[:80]

        with open(f"{out_dir}/{filename}.txt", "w", encoding="utf-8") as f:
            f.write(story)

        print(f"Saved: {filename}.txt")
        return filename

    except Exception as e:
        print(f"Error scraping {url}: {e}")

In [14]:
for url in urls:
    scrape_story(url)
    time.sleep(1)

Saved: The_Weight_of_Silence.txt
Saved: The_Bargain.txt
Saved: Waiting_for_a_Call.txt
Saved: The_Voice.txt
Saved: THE_MALLARD.txt
Saved: Not_the_Gift_of_the_Magi.txt
Saved: Unbound.txt
Saved: The_Ghost.txt
Saved: A_Study_in_Cashmere.txt
Saved: Duck_Eggs.txt
Saved: High.txt
Saved: Mama.txt
Saved: Shattered.txt
Saved: Faceless.txt
Saved: Dear_John.txt
Saved: Peccadillos.txt
Saved: Rose.txt
Saved: Love_Wants.txt
Saved: Divine_January_Pt__3.txt
Saved: The_Burden_s_Weight.txt
Saved: Divine_January_Pt__1.txt
Saved: Charm_and_Insecurity.txt
Saved: Missing__part_2.txt
Saved: I_Am_Not_A_Crook.txt
Saved: The_Right_Hand_Man.txt
Saved: The_Fall.txt
Saved: A_Good_Man_Is_Hard_to_Find_and_Other_Life_Lessons.txt
Saved: Seven_Seconds_to_Midnight.txt
Saved: Suburbia_is_an_Awkward_Place_for_Superheroes.txt
Saved: Nameless_52.txt
Saved: It_started_as_a_whisper.txt
Saved: The_Dinner_Party.txt
Saved: Shoes.txt
Saved: Give_the_Devil_His_Due.txt
Saved: Running_Scared.txt
Saved: The_Rise_Part_2__The_Outlier.tx